# Getting Started with Zipminator

This notebook demonstrates the core Zipminator workflow: generating Kyber768 keypairs,
encrypting and decrypting messages, and applying DataFrame anonymization.

**Prerequisites**: `micromamba activate zip-pqc && uv pip install -e '.[jupyter,data]'`

Optionally run `maturin develop` to use the faster Rust backend. Without it, the pure-Python
fallback (`kyber-py`) is used automatically.

In [1]:
# Generate a Kyber768 keypair
from zipminator import keypair, encapsulate, decapsulate, RUST_AVAILABLE

# Helper: Rust FFI returns custom types with .to_bytes(); Python returns raw bytes
def to_bytes(obj):
    return obj.to_bytes() if hasattr(obj, 'to_bytes') else obj

pk, sk = keypair()

print(f"Backend:          {'Rust (native)' if RUST_AVAILABLE else 'Python (kyber-py)'}")
print(f"Public key size:  {len(to_bytes(pk))} bytes")
print(f"Secret key size:  {len(to_bytes(sk))} bytes")
print(f"Public key (hex): {to_bytes(pk)[:16].hex()}...")

Backend:          Rust (native)
Public key size:  1184 bytes
Secret key size:  2400 bytes
Public key (hex): 5bb3947f7b3b922177c988bddbea077a...


In [2]:
# Encrypt and decrypt a message using KEM + AES-256-GCM
from hashlib import sha3_256
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# KEM encapsulation: sender gets ciphertext + shared secret
ct, shared_secret_sender = encapsulate(pk)
print(f"Ciphertext size:    {len(to_bytes(ct))} bytes")
print(f"Shared secret (tx): {shared_secret_sender.hex()[:32]}...")

# KEM decapsulation: receiver recovers the same shared secret
shared_secret_receiver = decapsulate(ct, sk)
print(f"Shared secret (rx): {shared_secret_receiver.hex()[:32]}...")
assert shared_secret_sender == shared_secret_receiver, "Shared secrets must match!"
print("Shared secrets match.")

# Derive AES-256 key from shared secret
aes_key = sha3_256(shared_secret_sender).digest()

# Encrypt a message
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)
message = b"Hello from the post-quantum world!"
ciphertext = aesgcm.encrypt(nonce, message, None)
print(f"\nPlaintext:  {message.decode()}")
print(f"Ciphertext: {ciphertext[:24].hex()}... ({len(ciphertext)} bytes)")

# Decrypt
plaintext = aesgcm.decrypt(nonce, ciphertext, None)
print(f"Decrypted:  {plaintext.decode()}")

Ciphertext size:    1088 bytes
Shared secret (tx): ee4eacd433f79de18070572a9bbdaf9f...
Shared secret (rx): ee4eacd433f79de18070572a9bbdaf9f...
Shared secrets match.

Plaintext:  Hello from the post-quantum world!
Ciphertext: d0d0688cba1111e71f3cea50d2692795a2f8ac5e4ad207ad... (50 bytes)
Decrypted:  Hello from the post-quantum world!


In [3]:
# IPython magics: interactive PQC in one line
%load_ext zipminator.jupyter
%zipminator_info

Version,0.5.0b1
Backend,RUST (native)
Algorithm,ML-KEM-768 (FIPS 203)
PK / SK / CT,1184 / 2400 / 1088 bytes
Shared secret,32 bytes
Entropy pool,"19,456 bytes"


In [4]:
# Magic round-trip
%keygen
%encrypt pk
%decrypt ct sk
print(f"Shared secrets match: {shared_secret == recovered}")

Public key,1184 bytes
Secret key,2400 bytes
PK preview,ac 2a c5 20 20 8e fe 99 49 13 53 63 60 49 2c 0b 12 42 08 b4 19 c2 b3 85 9d 87 49 4f 5a 23 d2 90 ...
Elapsed,0.06 ms
Entropy,SYSTEM RNG


Ciphertext,1088 bytes
Shared secret,32 bytes
CT preview,ce f2 46 b1 33 e0 d6 74 90 26 9c a2 01 ab cf da 22 55 8b 86 0d 63 de 57 b9 e5 34 19 21 b0 42 53 ...
SS preview,2a 13 5a ed d2 fd 03 17 59 aa 8c 26 8f d5 9e f6 68 5c 68 6f 7e 7f 8c 9e b9 e5 8a e1 6e 10 e8 5e
Elapsed,0.08 ms


Shared secret,32 bytes
SS preview,2a 13 5a ed d2 fd 03 17 59 aa 8c 26 8f d5 9e f6 68 5c 68 6f 7e 7f 8c 9e b9 e5 8a e1 6e 10 e8 5e
Elapsed,0.08 ms


Shared secrets match: True


In [ ]:
# DataFrame anonymization with PII scanning
import pandas as pd
from zipminator.anonymizer import AdvancedAnonymizer

anonymizer = AdvancedAnonymizer()

df = pd.DataFrame({
    "name": ["Alice Johnson", "Bob Smith", "Carol Williams"],
    "email": ["alice@example.com", "bob@corp.net", "carol@hospital.org"],
    "phone": ["555-123-4567", "555-987-6543", "555-246-8135"],
    "salary": [85000, 92000, 78000],
})

print("Original DataFrame:")
print(df.to_string(index=False))
print()

# Anonymize: level_map assigns anonymization level per column
level_map = {col: 1 for col in df.columns}
result = anonymizer.process(df, level_map)
print("Level 1 Anonymized (Minimal Masking):")
print(result.to_string(index=False))

In [ ]:
# Interactive widgets
from zipminator.jupyter.widgets import key_size_comparison, entropy_monitor
key_size_comparison()